# DuckDB Queries

In [1]:
from pathlib import Path
import duckdb
from pyiceberg.catalog.sql import SqlCatalog

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / "data"
SQL_DIR = PROJECT_ROOT / "sql"

catalog = SqlCatalog(
    "local",
    uri=f"sqlite:///{(DATA_DIR / 'catalog.db').resolve()}",
    warehouse=(DATA_DIR / "warehouse").resolve().as_uri(),
)
con = duckdb.connect(database=":memory:")


In [2]:
TABLES = {
    "daily_trip_summary": "gold.daily_trip_summary",
    "monthly_kpi": "gold.monthly_kpi",
    "zone_flow_monthly": "gold.zone_flow_monthly",
    "payment_monthly": "gold.payment_monthly",
    "vendor_monthly": "gold.vendor_monthly",
    "data_quality_summary": "gold.data_quality_summary",
    "payment_null_profile": "gold.payment_null_profile",
}
for duckdb_name, iceberg_name in TABLES.items():
    con.register(duckdb_name, catalog.load_table(iceberg_name).scan().to_arrow())
con.sql("SHOW TABLES").df()


,name
0,daily_trip_summary
1,data_quality_summary
2,monthly_kpi
3,payment_monthly
4,payment_null_profile
5,vendor_monthly
6,zone_flow_monthly


In [3]:
monthly_sql = (SQL_DIR / "monthly_kpi.sql").read_text()
con.execute(
    monthly_sql,
    {"data_variant": "all", "month_basis": "pickup_month"},
).df()


,data_variant,month_basis,year_month,trip_count,passenger_count_total,total_trip_distance,total_revenue,total_fare_amount,total_tip_amount,avg_fare_amount,avg_total_amount,avg_trip_distance,avg_trip_duration_min,avg_speed_mph
0,all,pickup_month,2025-12,5,7,4.426000e+01,3.298500e+02,2.756000e+02,4.00,55.120000,65.970000,8.852000,20.700000,14.463352
1,all,pickup_month,2026-01,3670081,3259306,2.384942e+07,1.095242e+08,7.821451e+07,9668940.07,21.311386,29.842455,6.498336,17.226838,24.433316
2,all,pickup_month,2026-02,3359632,2887682,2.108837e+07,1.028571e+08,7.401797e+07,8932198.50,22.031572,30.615583,6.276987,17.869095,23.723080
3,all,pickup_month,2026-03,3918080,3672922,2.354446e+07,1.192198e+08,8.454436e+07,11252367.02,21.578008,30.428124,6.009184,17.316782,21.442874
4,all,pickup_month,2026-04,2,6,4.840000e+00,4.543000e+01,2.910000e+01,4.83,14.550000,22.715000,2.420000,11.766667,12.383626


In [4]:
con.execute(
    monthly_sql,
    {"data_variant": "all", "month_basis": "source_month"},
).df()


,data_variant,month_basis,year_month,trip_count,passenger_count_total,total_trip_distance,total_revenue,total_fare_amount,total_tip_amount,avg_fare_amount,avg_total_amount,avg_trip_distance,avg_trip_duration_min,avg_speed_mph
0,all,source_month,2026-01,3670075,3259301,2.384944e+07,1.095243e+08,7.821465e+07,9668917.51,21.311458,29.842531,6.498353,17.226863,24.433348
1,all,source_month,2026-02,3359632,2887669,2.108836e+07,1.028570e+08,7.401790e+07,8932179.49,22.031550,30.615556,6.276985,17.869082,23.723087
2,all,source_month,2026-03,3918093,3672953,2.354450e+07,1.192202e+08,8.454461e+07,11252417.42,21.577999,30.428116,6.009172,17.316772,21.442831


In [5]:
zone_sql = (SQL_DIR / "zone_analysis.sql").read_text()
con.execute(
    zone_sql,
    {
        "data_variant": "all",
        "month_basis": "pickup_month",
        "year_month": None,
        "pickup_location_id": 132,
        "dropoff_location_id": None,
        "row_limit": 25,
    },
).df()


,data_variant,month_basis,year_month,pickup_location_id,dropoff_location_id,trip_count,total_revenue,avg_total_amount,avg_fare_amount,avg_tip_amount,avg_trip_distance,avg_trip_duration_min,avg_speed_mph
0,all,pickup_month,2026-01,132,265,7137,877640.98,122.970573,102.817039,10.922219,20.388843,39.181113,30.980141
1,all,pickup_month,2026-03,132,265,6548,825902.59,126.130512,104.722692,12.278348,20.476263,40.932476,29.862282
2,all,pickup_month,2026-03,132,132,6527,360095.32,55.170112,45.879683,5.150149,1.508027,7.425821,23.384193
3,all,pickup_month,2026-01,132,132,6080,316108.17,51.991475,43.107252,4.835469,1.538258,7.888613,18.432215
4,all,pickup_month,2026-02,132,265,6029,724596.99,120.185270,100.444712,11.215210,19.753394,38.962274,30.526567
5,all,pickup_month,2026-03,132,230,5668,552600.68,97.494827,70.804248,11.838520,17.603629,56.684395,19.987024
6,all,pickup_month,2026-02,132,132,5081,268261.65,52.797018,43.868351,4.971510,1.606684,8.084281,21.879350
7,all,pickup_month,2026-01,132,230,4898,476384.28,97.260980,71.135945,11.458303,17.660958,54.826085,20.884784
8,all,pickup_month,2026-03,132,93,4331,156575.78,36.152339,31.739171,0.155036,8.013856,16.051093,52.616257
9,all,pickup_month,2026-02,132,230,4251,411325.87,96.759791,70.988236,11.155034,17.648908,57.182777,19.838206


In [6]:
payment_sql = (SQL_DIR / "payment_analysis.sql").read_text()
con.execute(
    payment_sql,
    {
        "data_variant": "all",
        "month_basis": "source_month",
        "year_month": None,
        "payment_type": None,
    },
).df()


,data_variant,month_basis,year_month,payment_type,trip_count,total_revenue,total_fare_amount,total_tip_amount,avg_total_amount,avg_fare_amount,avg_tip_amount,recorded_tip_to_fare_rate,avg_trip_distance,avg_trip_duration_min
0,all,source_month,2026-01,0,1087979,3.446928e+07,2.761057e+07,408051.81,31.681939,25.377848,0.375055,0.014779,13.749030,17.861661
1,all,source_month,2026-01,1,2236989,6.629434e+07,4.379478e+07,9260650.23,29.635524,19.577556,4.139784,0.211456,3.439425,17.202877
2,all,source_month,2026-01,2,303050,7.603481e+06,5.898989e+06,122.75,25.089858,19.465398,0.000405,0.000021,3.462052,15.585695
3,all,source_month,2026-01,3,11806,2.722710e+05,2.085766e+05,2.60,23.062087,17.666996,0.000220,0.000012,2.530756,11.049478
4,all,source_month,2026-01,4,30251,8.849499e+05,7.017386e+05,90.12,29.253576,23.197204,0.002979,0.000128,3.893328,15.021818
5,all,source_month,2026-02,0,1023230,3.446237e+07,2.785882e+07,421297.07,33.679982,27.226353,0.411733,0.015123,12.967393,18.369869
6,all,source_month,2026-02,1,2040579,6.084634e+07,4.027703e+07,8510638.02,29.818174,19.738040,4.170698,0.211303,3.349818,17.865910
7,all,source_month,2026-02,2,265495,6.743821e+06,5.253244e+06,69.28,25.400935,19.786603,0.000261,0.000013,3.336585,16.362828
8,all,source_month,2026-02,3,9126,2.022424e+05,1.534100e+05,14.38,22.161124,16.810209,0.001576,0.000094,2.352128,11.452933
9,all,source_month,2026-02,4,21202,6.022311e+05,4.753952e+05,160.74,28.404446,22.422187,0.007581,0.000338,3.624844,15.629118


In [7]:
quality_sql = (SQL_DIR / "data_quality.sql").read_text()
con.execute(
    quality_sql,
    {
        "month_basis": "overall",
        "metric_category": "silver_warning",
        "source_month": None,
    },
).df()


,month_basis,metric_date,source_file,source_month,metric_category,metric_name,metric_value,metric_percent,percent_denominator,generated_at
0,overall,2026-07-12,None,None,silver_warning,total_less_than_fare,7363.0,0.067256,silver_rows,2026-07-13 00:22:09.213854+07:00
1,overall,2026-07-12,None,None,silver_warning,zero_distance,361150.0,3.298836,silver_rows,2026-07-13 00:22:09.213854+07:00
2,overall,2026-07-12,None,None,silver_warning,zero_duration,134516.0,1.228703,silver_rows,2026-07-13 00:22:09.213854+07:00
3,overall,2026-07-12,None,None,silver_warning,zero_fare_amount,5693.0,0.052001,silver_rows,2026-07-13 00:22:09.213854+07:00
4,overall,2026-07-12,None,None,silver_warning,zero_total_amount,1470.0,0.013427,silver_rows,2026-07-13 00:22:09.213854+07:00


In [8]:
con.execute(
    quality_sql,
    {
        "month_basis": "pickup_month",
        "metric_category": "zero_filter_impact_month",
        "source_month": None,
    },
).df()


,month_basis,metric_date,source_file,source_month,metric_category,metric_name,metric_value,metric_percent,percent_denominator,generated_at
0,pickup_month,2026-07-12,None,2025-12,zero_filter_impact_month,zero_distance,0.0,0.000000,silver_rows_in_month,2026-07-13 00:22:09.213854+07:00
1,pickup_month,2026-07-12,None,2025-12,zero_filter_impact_month,zero_duration,0.0,0.000000,silver_rows_in_month,2026-07-13 00:22:09.213854+07:00
2,pickup_month,2026-07-12,None,2025-12,zero_filter_impact_month,zero_duration_and_distance,0.0,0.000000,silver_rows_in_month,2026-07-13 00:22:09.213854+07:00
3,pickup_month,2026-07-12,None,2025-12,zero_filter_impact_month,zero_duration_or_distance,0.0,0.000000,silver_rows_in_month,2026-07-13 00:22:09.213854+07:00
4,pickup_month,2026-07-12,None,2026-01,zero_filter_impact_month,zero_distance,122295.0,3.332215,silver_rows_in_month,2026-07-13 00:22:09.213854+07:00
5,pickup_month,2026-07-12,None,2026-01,zero_filter_impact_month,zero_duration,44961.0,1.225068,silver_rows_in_month,2026-07-13 00:22:09.213854+07:00
6,pickup_month,2026-07-12,None,2026-01,zero_filter_impact_month,zero_duration_and_distance,1057.0,0.028800,silver_rows_in_month,2026-07-13 00:22:09.213854+07:00
7,pickup_month,2026-07-12,None,2026-01,zero_filter_impact_month,zero_duration_or_distance,166199.0,4.528483,silver_rows_in_month,2026-07-13 00:22:09.213854+07:00
8,pickup_month,2026-07-12,None,2026-02,zero_filter_impact_month,zero_distance,120346.0,3.582119,silver_rows_in_month,2026-07-13 00:22:09.213854+07:00
9,pickup_month,2026-07-12,None,2026-02,zero_filter_impact_month,zero_duration,40470.0,1.204596,silver_rows_in_month,2026-07-13 00:22:09.213854+07:00


In [9]:
profile_sql = (SQL_DIR / "payment_null_profile.sql").read_text()
con.execute(
    profile_sql,
    {"data_variant": "all", "payment_type": 0},
).df()


,data_variant,payment_type,field_name,row_count,null_count,null_percent,all_profile_fields_null_count,all_profile_fields_null_percent
0,all,0,airport_fee,3056852,3056852,100.0,3056852,100.0
1,all,0,congestion_surcharge,3056852,3056852,100.0,3056852,100.0
2,all,0,passenger_count,3056852,3056852,100.0,3056852,100.0
3,all,0,ratecode_id,3056852,3056852,100.0,3056852,100.0
4,all,0,store_and_fwd_flag,3056852,3056852,100.0,3056852,100.0


In [10]:
con.close()
